In [46]:
# 1-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/4_1_1.csv')
# print(df.info())
# print(df.head())
df.dropna(inplace=True)
# print(df.info())
# print(df.head())
a = int(len(df) * 0.7)
df1 = df[:a]
df1['PTRATIO'].quantile(.25).round(1)

np.float64(16.4)

In [69]:
# 1-2
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/4_1_2.csv')
# print(df.info())
# print(df.head())
a = df['avgSchoolRating'].mean()
rs1 = df[(df['yearBuilt'] >= 1991) & (df['yearBuilt'] <= 2000) & (df['avgSchoolRating'] <= a)]['uid']
rs2 = df[(df['yearBuilt'] >= 2001) & (df['yearBuilt'] <= 2010) & (df['avgSchoolRating'] >= a)]['uid']
rs3 = pd.concat([rs1,rs2], axis=0)
print(len(rs3.unique()))

76


In [54]:
# 1-3
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/4_1_3.csv')
print(df.info())

HTTPError: HTTP Error 404: Not Found

In [127]:
# 2
import numpy as np
import pandas as pd
train_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/4_2_train.csv')
test_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/4_2_test.csv')
print(train_df.info())
# print(train_df.head())
# print(test_df.info())

from sklearn.metrics import get_scorer_names, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
# roc_auc

train_X = train_df.drop(columns=['ID', 'Attrition_Flag'])
# print(train_X)
train_y = train_df.pop('Attrition_Flag')
# print(train_y)
test_X = test_df.drop(columns=['ID', 'Attrition_Flag'])
test_y = test_df.pop('Attrition_Flag')

train_y = train_y.map({'Existing Customer': 0, 'Attrited Customer': 1})
test_y = test_y.map({'Existing Customer': 0, 'Attrited Customer': 1})

train_X, valid_X, train_y, valid_y = train_test_split(train_X, train_y, test_size=0.3)
print(train_X.shape, train_y.shape, valid_X.shape, valid_y.shape)

train_X_num = train_X.select_dtypes('number').columns.tolist()
train_X_obj = train_X.select_dtypes('object').columns.tolist()
valid_X_num = valid_X.select_dtypes('number').columns.tolist()
valid_X_obj = valid_X.select_dtypes('object').columns.tolist()
test_X_num = test_X.select_dtypes('number').columns.tolist()
test_X_obj = test_X.select_dtypes('object').columns.tolist()

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
train_X_encoder = encoder.fit_transform(train_X[train_X_obj])
valid_X_encoder = encoder.transform(valid_X[valid_X_obj])
test_X_encoder = encoder.transform(test_X[test_X_obj])

train_X = np.concatenate([train_X_encoder, train_X[train_X_num]], axis=1)
valid_X = np.concatenate([valid_X_encoder, valid_X[valid_X_num]], axis=1)
test_X = np.concatenate([test_X_encoder, test_X[test_X_num]], axis=1)

rf = RandomForestClassifier()
rf.fit(train_X, train_y)
pred1 = rf.predict_proba(valid_X)[:,1]
# pred1
print(roc_auc_score(valid_y, pred1))

params={
    'max_depth': [10,20,30],
    'min_samples_split': [2,5,10]
}
cv = GridSearchCV(estimator=rf, param_grid=params, scoring='roc_auc', cv=3)
cv.fit(train_X, train_y)
pred2 = cv.predict_proba(valid_X)[:,1]
print(roc_auc_score(valid_y, pred2))

pred3 = cv.predict_proba(test_X)[:,1]
pred4 = rf.predict_proba(test_X)[:,1]
pd.DataFrame(pred3, columns=['pred']).to_csv('result.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354 entries, 0 to 353
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ID                      354 non-null    int64  
 1   Attrition_Flag          354 non-null    object 
 2   Gender                  354 non-null    object 
 3   Customer_Age            354 non-null    int64  
 4   Income_Category         354 non-null    object 
 5   Card_Category           354 non-null    object 
 6   Credit_Limit            354 non-null    float64
 7   Total_Revolving_Bal     354 non-null    int64  
 8   Avg_Utilization_Ratio   354 non-null    float64
 9   Avg_Open_To_Buy         354 non-null    float64
 10  Months_Inactive_12_mon  354 non-null    int64  
dtypes: float64(3), int64(4), object(4)
memory usage: 30.6+ KB
None
(247, 9) (247,) (107, 9) (107,)
0.7644736842105263
0.8105263157894737


In [21]:
# 3-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/4_3_1.csv')
# print(df.info())
# print(df.head())

print((df.groupby('department')['hours_worked'].mean()['A'] + df.groupby('department')['hours_worked'].mean()['B']).round(2))
print((df.groupby('department')['hours_worked'].std()['A'] + df.groupby('department')['hours_worked'].std()['B']).round(2))

from scipy.stats import ttest_ind
a = df[df['department'] == 'A']['hours_worked']
b = df[df['department'] == 'B']['hours_worked']
print(ttest_ind(a, b).statistic.round(2))
print(ttest_ind(a, b).pvalue.round(3) < 0.05)

85.77
10.07
-1.02
False


In [36]:
# 3-2
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/4_3_2.csv')
# print(df.info())
# print(df.head())
df.groupby('factory')['quality_score'].mean()
print((df.groupby('factory')['quality_score'].mean()['A'] + df.groupby('factory')['quality_score'].mean()['B'] + df.groupby('factory')['quality_score'].mean()['C']).round(2))
print((df.groupby('factory')['quality_score'].std()['A'] + df.groupby('factory')['quality_score'].std()['B'] + df.groupby('factory')['quality_score'].std()['C']).round(2))

from scipy.stats import kruskal
a = df[df['factory']=='A']['quality_score']
b = df[df['factory']=='B']['quality_score']
c = df[df['factory']=='C']['quality_score']
print(kruskal(a,b,c).statistic.round(2))
print(kruskal(a,b,c).pvalue.round(3) < 0.05)

225.2
29.8
5.93
False
